# Nemotron RFT — Generate model's own correct CoTs (vLLM, 100% offline)

**Goal:** lift LB from 0.86 → ~0.88–0.90 by training on the model's own
correct reasoning instead of the noisy original CoT.

**Engine:** vLLM with continuous batching + LoRA. ~5-10× faster than HF
`model.generate()`.

**Source of vLLM:** the **NVIDIA metric utility script**
(`/kaggle/usr/lib/nvidia-metric-utility-script`), which ships vLLM, Mamba
CUDA kernels, Triton, and all CUDA libs pre-built for Kaggle's RTX 6000
Pro Blackwell environment. **No internet required.**

This is the same install path the official baseline-evaluation notebook
uses (`nemotron-baseline-evaluation.ipynb`).

## Pipeline (3 stages, single notebook)

1. **Generate** — load 0.86 adapter into vLLM, sample K=4 completions per
   prompt on hard categories (bit_manipulation, equation_numeric_*,
   cryptarithm_*) in ONE batched call.

2. **Filter** — extract `\boxed{...}`, compare to ground truth, drop wrong
   + duplicate. Saves to `/kaggle/working/rft_train.jsonl`.

3. **Stats** — yield per category.

## Required Kaggle inputs
- `metric/nemotron-3-nano-30b-a3b-bf16` (model)
- The **NVIDIA Nemotron metric utility script** (auto-attached when you
  enable the competition metric; appears at
  `/kaggle/usr/lib/nvidia-metric-utility-script`)
- **Your 0.86 adapter dataset** — set `ADAPTER_PATH` in cell 4
- `nemotron-categorical-splits` (your training data)

## Settings
- **Internet: OFF** ← all installs come from the metric utility script
- Accelerator: GPU (RTX 6000 Pro Blackwell)

## Expected runtime
- vLLM engine boot: ~3–4 min
- Generation: **~30–60 min** for ~1500 prompts × K=4 samples
- Filter + stats: <1 min


In [ ]:
# ============================================================
# 1. SETUP — extract bundle that ships vLLM (offline)
# ============================================================
# The tinker-submission-notebook AND the official baseline-evaluation
# notebook both extract this exact bundle and successfully import vLLM
# from /tmp/vllm/. Bundle path uses UNDERSCORES not hyphens, and lives
# under notebooks/metric/.
import subprocess, sys, os, glob, importlib

# Prefer the bundle that the working notebooks use (it ships vLLM).
# Fall back to other locations if Kaggle reorganized things.
CANDIDATE_BUNDLES = [
    "/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script",  # ← has vLLM
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script",
    "/kaggle/usr/lib/nvidia-metric-utility-script",                   # stripped, no vLLM
]
bundle = next((b for b in CANDIDATE_BUNDLES if os.path.isdir(b)), None)
if bundle is None:
    raise FileNotFoundError(
        "No NVIDIA utility-script bundle found. Searched:\n  "
        + "\n  ".join(CANDIDATE_BUNDLES) +
        "\nEnable the competition metric so the bundle auto-attaches."
    )
print(f"[ok] Using bundle: {bundle}")

# Step 1: kill Kaggle's pre-installed torch (METH_CLASS crash with bundle's
# transformers 5.3 if both are loadable)
subprocess.run(
    "uv pip uninstall torch torchvision torchaudio || "
    "pip uninstall -y torch torchvision torchaudio || true",
    shell=True, check=False
)

# Step 2: extract bundle (flat layout — packages land directly in /tmp/)
print(f"Extracting bundle to /tmp ...")
r = subprocess.run(
    f"tar -cf - -C {bundle} . | tar -xf - -C /tmp",
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    print(f"[warn] tar stderr: {r.stderr[:300]}")

# Step 3: ptxas binaries
for ptxas in ["/tmp/triton/backends/nvidia/bin/ptxas",
              "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"]:
    if os.path.exists(ptxas):
        subprocess.run(f"chmod +x {ptxas}", shell=True)

# Step 4: confirm vLLM is in /tmp
vllm_init = "/tmp/vllm/__init__.py"
if not os.path.exists(vllm_init):
    # search for it elsewhere
    found = glob.glob("/tmp/**/vllm/__init__.py", recursive=True) + \
            glob.glob("/kaggle/usr/lib/**/vllm/__init__.py", recursive=True)
    raise ImportError(
        f"vLLM not found at /tmp/vllm/.\n"
        f"Other vllm/ locations: {found}\n"
        f"You probably attached the wrong bundle. The one with vLLM is at:\n"
        f"  /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script\n"
        f"Enable the competition metric (auto-attaches it) and restart the kernel."
    )
print(f"[ok] vLLM present at /tmp/vllm/")

# Step 5: prune sys.path of competing locations and put /tmp FIRST
PRUNE_HINTS = ("ryanholbrook/nvidia_utility_script", "/kaggle/working/packages")
sys.path = [p for p in sys.path if not any(h in p for h in PRUNE_HINTS)]
sys.path = ["/tmp"] + [p for p in sys.path if p != "/tmp"]
print(f"sys.path[:3] = {sys.path[:3]}")

# Step 6: evict cached modules so /tmp versions win
for _m in list(sys.modules):
    top = _m.split(".")[0]
    if top in (
        "torch", "torchvision", "torchaudio", "torchgen", "functorch",
        "transformers", "tokenizers", "safetensors", "huggingface_hub",
        "accelerate", "peft", "datasets", "triton",
        "mamba_ssm", "causal_conv1d", "flash_attn", "vllm",
    ):
        del sys.modules[_m]

# Step 7: verify imports come from /tmp
print("\nVerifying imports:")
import torch
print(f"  torch        {torch.__version__:25s}  {torch.__file__}")
import transformers
print(f"  transformers {transformers.__version__:25s}  {transformers.__file__}")
import vllm
print(f"  vllm         {vllm.__version__:25s}  {vllm.__file__}")

assert torch.__file__.startswith("/tmp/"),       f"torch not from /tmp: {torch.__file__}"
assert transformers.__file__.startswith("/tmp/"), f"transformers not from /tmp: {transformers.__file__}"
assert vllm.__file__.startswith("/tmp/"),         f"vllm not from /tmp: {vllm.__file__}"

print(f"\n[ok] cell 1 done — vLLM {vllm.__version__}, torch {torch.__version__}, "
      f"CUDA: {torch.cuda.is_available()}")

In [2]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, time, re, hashlib
from collections import Counter, defaultdict
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file as load_safetensors

print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
import transformers
print(f"transformers : {transformers.__version__}")

# Mamba fast path check
try:
    import causal_conv1d, mamba_ssm
    print(f"causal_conv1d: {causal_conv1d.__version__}  ← Mamba fast path ON")
    print(f"mamba_ssm    : {mamba_ssm.__version__}")
except ImportError as e:
    print(f"[warn] Mamba fast path OFF — {e}")

PyTorch      : 2.12.0.dev20260324+cu128
GPU          : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM         : 102.0 GB
transformers : 5.3.0
[warn] Mamba fast path OFF — No module named 'cutlass'


In [3]:
# ============================================================
# 3. TRITON ENV — point at extracted ptxas (offline)
# ============================================================
# The utility script extraction in cell 1 already put ptxas-blackwell at
# /tmp/triton/backends/nvidia/bin/ptxas-blackwell. Just point env vars at it.
import os
PTXAS_PATH = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"
if os.path.exists(PTXAS_PATH):
    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
              "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
        os.environ[v] = PTXAS_PATH
    print(f"[ok] ptxas pointed at {PTXAS_PATH}")
else:
    # Fall back to non-Blackwell ptxas
    PTXAS_PATH = "/tmp/triton/backends/nvidia/bin/ptxas"
    if os.path.exists(PTXAS_PATH):
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_PATH
        print(f"[ok] ptxas (non-blackwell) pointed at {PTXAS_PATH}")
    else:
        print(f"[warn] no ptxas found in /tmp/triton")


[ok] ptxas pointed at /tmp/triton/backends/nvidia/bin/ptxas-blackwell


In [4]:
# ============================================================
# 4. CONFIG — paths, generation params, target categories
# ============================================================
MODEL_PATH    = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# *** EDIT THIS *** — path to YOUR 0.86 adapter dataset
ADAPTER_PATH  = "/kaggle/input/models/manish756/nvidia-adapter/transformers/default/7"

# Where the 9 category JSONLs live
DATA_DIR_CANDIDATES = [
    "/kaggle/input/datasets/manish756/nemotron-dataset/all_categorical_splits",
]

# RFT focuses on the categories where the model has room to improve.
# The "easy" ones (cipher, unit_conversion, numeral, gravity) already train
# well from the original CoTs — no need to spend rollout compute on them.
HARD_CATEGORIES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
]

# Generation settings
K            = 4         # samples per prompt
TEMPERATURE  = 0.7
TOP_P        = 0.95
MAX_NEW_TOK  = 4096      # enough headroom for full CoT (the model's CoTs are
                         # typically shorter than the original training CoTs)

RAW_OUTPUT      = "/kaggle/working/raw_generations.jsonl"
RFT_TRAIN_FILE  = "/kaggle/working/rft_train.jsonl"

# Optional: cap how many prompts to process per category (for fast experiments).
# Set to None for full run.
MAX_PROMPTS_PER_CATEGORY = None

print(f"Adapter        : {ADAPTER_PATH}")
print(f"Hard cats      : {HARD_CATEGORIES}")
print(f"K              : {K}    temp={TEMPERATURE}  top_p={TOP_P}  max_new={MAX_NEW_TOK}")
print(f"Output         : {RAW_OUTPUT}")
print(f"Per-cat cap    : {MAX_PROMPTS_PER_CATEGORY or 'unlimited'}")


Adapter        : /kaggle/input/models/manish756/nvidia-adapter/transformers/default/7
Hard cats      : ['train_cot_bit_manipulation.jsonl', 'train_cot_equation_numeric_deduce.jsonl', 'train_cot_cryptarithm_deduce.jsonl', 'train_cot_cryptarithm_guess.jsonl', 'train_cot_equation_numeric_guess.jsonl']
K              : 4    temp=0.7  top_p=0.95  max_new=4096
Output         : /kaggle/working/raw_generations.jsonl
Per-cat cap    : unlimited


In [ ]:
# ============================================================
# 5. LOAD vLLM ENGINE + LoRA ADAPTER
# ============================================================
# vLLM with continuous batching + LoRARequest. Same exact pattern as the
# tinker-submission-notebook and baseline-evaluation. ~5-10× faster than
# HF generate(). Expected: ~30-60 min for 2800 prompts × K=4.
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer

# Verify adapter
adapter_cfg = os.path.join(ADAPTER_PATH, "adapter_config.json")
adapter_w   = os.path.join(ADAPTER_PATH, "adapter_model.safetensors")
assert os.path.exists(adapter_cfg) and os.path.exists(adapter_w), (
    f"adapter files not found at {ADAPTER_PATH}\n"
    "Edit ADAPTER_PATH in cell 4."
)
with open(adapter_cfg) as f:
    cfg = json.load(f)
print(f"Adapter:  r={cfg.get('r')}  alpha={cfg.get('lora_alpha')}  "
      f"target={cfg.get('target_modules')}")
LORA_RANK_FROM_CFG = cfg.get("r", 32)

# Tokenizer (used to chat-template prompts before passing strings to vLLM)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# vLLM's torch.compile uses triton → point at extracted ptxas
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

print("\nLoading vLLM engine (this takes ~3-4 min) ...")
llm = LLM(
    model                  = MODEL_PATH,
    trust_remote_code      = True,
    dtype                  = "bfloat16",
    enable_lora            = True,
    max_lora_rank          = max(LORA_RANK_FROM_CFG, 32),
    max_loras              = 1,
    max_num_seqs           = 64,         # continuous-batching width
    gpu_memory_utilization = 0.85,
    max_model_len          = 8192,
    enable_prefix_caching  = True,
    enable_chunked_prefill = True,
)

# LoRARequest — pass to every generate() call to apply the adapter
lora_req = LoRARequest(
    lora_name   = "rft_adapter",
    lora_int_id = 1,
    lora_path   = ADAPTER_PATH,
)

print("\nSanity test (1 prompt, deterministic, 32 tokens):")
sp_test = SamplingParams(n=1, temperature=0.0, max_tokens=32)
test_outputs = llm.generate(
    ["Say hello in three words."],
    sampling_params = sp_test,
    lora_request    = lora_req,
)
print(f"  output: {test_outputs[0].outputs[0].text!r}")
print("[ok] vLLM engine + LoRA adapter ready")

In [6]:
# ============================================================
# 6. LOAD PROMPTS (hard categories only) + ground-truth answers
# ============================================================
data_dir = None
for c in DATA_DIR_CANDIDATES:
    if c and os.path.isdir(c) and any(
        os.path.exists(os.path.join(c, f)) for f in HARD_CATEGORIES
    ):
        data_dir = c; break
assert data_dir, f"no data dir found; searched: {DATA_DIR_CANDIDATES}"
print(f"Data dir: {data_dir}")

def extract_boxed(text):
    """Return the LAST \\boxed{...} content. None if missing."""
    matches = re.findall(r"\\boxed\{([^}]*)\}", text)
    return matches[-1].strip() if matches else None

prompts = []
for fname in HARD_CATEGORIES:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [skip] {fname}"); continue
    cat = fname.replace("train_cot_", "").replace(".jsonl", "")
    n_loaded = 0; n_skipped = 0
    with open(fpath) as f:
        for line in f:
            if not line.strip(): continue
            r = json.loads(line)
            msgs = [m for m in r["messages"] if m["role"] != "system"]
            if not msgs or msgs[-1]["role"] != "assistant":
                n_skipped += 1; continue
            user = msgs[0]["content"]
            asst_orig = msgs[-1]["content"]
            gt = extract_boxed(asst_orig)
            if gt is None:
                n_skipped += 1; continue
            prompts.append({
                "category": cat,
                "user": user,
                "gt_answer": gt,
            })
            n_loaded += 1
            if MAX_PROMPTS_PER_CATEGORY and n_loaded >= MAX_PROMPTS_PER_CATEGORY:
                break
    print(f"  {n_loaded:>5} from {fname}  (skipped {n_skipped})")

# Dedupe by user prompt — RFT only needs unique prompts
seen = set(); unique = []
for p in prompts:
    key = hashlib.md5(p["user"].encode()).hexdigest()
    if key in seen: continue
    seen.add(key); unique.append(p)
print(f"\nTotal: {len(prompts)} → {len(unique)} unique prompts after dedup")
prompts = unique

# Per-category breakdown
cat_counts = Counter(p["category"] for p in prompts)
print("\nPer-category prompt counts:")
for c, n in cat_counts.most_common():
    print(f"  {c:30s} {n:>5}")

# Estimated generation time
SEC_PER_GEN = 5  # rough estimate at seq up to 4k tokens on RTX 6000 Pro
est_hrs = len(prompts) * K * SEC_PER_GEN / 3600
print(f"\nEstimated runtime: {len(prompts)} prompts × K={K} × ~{SEC_PER_GEN}s ≈ {est_hrs:.1f} hrs")


Data dir: /kaggle/input/datasets/manish756/nemotron-dataset/all_categorical_splits
   2728 from train_cot_bit_manipulation.jsonl  (skipped 0)
    540 from train_cot_equation_numeric_deduce.jsonl  (skipped 0)
    659 from train_cot_cryptarithm_deduce.jsonl  (skipped 0)
    164 from train_cot_cryptarithm_guess.jsonl  (skipped 0)
    111 from train_cot_equation_numeric_guess.jsonl  (skipped 0)

Total: 4202 → 2838 unique prompts after dedup

Per-category prompt counts:
  bit_manipulation                1364
  cryptarithm_deduce               659
  equation_numeric_deduce          540
  cryptarithm_guess                164
  equation_numeric_guess           111

Estimated runtime: 2838 prompts × K=4 × ~5s ≈ 15.8 hrs


In [ ]:
# ============================================================
# 7. GENERATE — vLLM batched inference (one call for all prompts)
# ============================================================
# vLLM's continuous batching schedules all prompts together and processes
# K=4 samples per prompt in parallel. For ~2800 prompts × K=4 = 11,200
# completions, expect ~30-90 min depending on output length.

# Resume support
done_idx = set()
if os.path.exists(RAW_OUTPUT):
    with open(RAW_OUTPUT) as f:
        for line in f:
            try: done_idx.add(json.loads(line)["prompt_idx"])
            except: pass
    print(f"Resuming: {len(done_idx)} done, {len(prompts)-len(done_idx)} remaining")
else:
    print(f"Fresh run: {len(prompts)} prompts")

# Build templated prompt strings, remember original indices
todo_prompts = []
todo_indices = []
for i, p in enumerate(prompts):
    if i in done_idx:
        continue
    msgs = [{"role": "user", "content": p["user"]}]
    try:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=True,   # Nemotron-H reasoning mode
        )
    except Exception:
        text = (f"<|im_start|>user\n{p['user']}<|im_end|>\n"
                f"<|im_start|>assistant\n")
    todo_prompts.append(text)
    todo_indices.append(i)

if not todo_prompts:
    print("Nothing to generate. Skip to cell 8.")
else:
    print(f"\nGenerating {len(todo_prompts)} prompts × K={K} samples each ...")
    print(f"Sampling: temp={TEMPERATURE}  top_p={TOP_P}  max_tokens={MAX_NEW_TOK}")

    sp = SamplingParams(
        n           = K,
        temperature = TEMPERATURE,
        top_p       = TOP_P,
        max_tokens  = MAX_NEW_TOK,
    )

    # The big call — vLLM handles all batching, scheduling, KV cache
    t_start = time.time()
    all_outputs = llm.generate(
        todo_prompts,
        sampling_params = sp,
        lora_request    = lora_req,
    )
    elapsed = time.time() - t_start
    print(f"\n[ok] vLLM generation done in {elapsed/60:.1f} min "
          f"({len(todo_prompts) * K / max(elapsed, 1):.1f} completions/sec)")

    # Write results
    n_written = 0
    with open(RAW_OUTPUT, "a") as out:
        for orig_idx, vllm_out in zip(todo_indices, all_outputs):
            p = prompts[orig_idx]
            completions = [o.text for o in vllm_out.outputs]
            rec = {
                "prompt_idx":  orig_idx,
                "category":    p["category"],
                "user":        p["user"],
                "gt_answer":   p["gt_answer"],
                "completions": completions,
            }
            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_written += 1
    print(f"Wrote {n_written} new records to {RAW_OUTPUT}")

In [ ]:
# ============================================================
# 8. FILTER — extract \boxed{}, compare to GT, dedupe, save SFT JSONL
# ============================================================
def normalize_answer(s):
    """Loose match — strip whitespace, case, surrounding quotes."""
    if s is None: return ""
    s = str(s).strip().lower()
    s = s.strip("\"' ")
    s = s.replace(" ", "")
    return s

stats = Counter()
seen_pairs = set()
keep = []

with open(RAW_OUTPUT) as f:
    for line in f:
        try:
            r = json.loads(line)
        except Exception:
            stats["bad_json"] += 1; continue

        gt = normalize_answer(r["gt_answer"])
        cat = r["category"]
        stats[f"prompt_{cat}"] += 1

        for comp in r["completions"]:
            stats["total_completions"] += 1
            pred = extract_boxed(comp)
            if pred is None:
                stats["no_box"] += 1; continue
            if normalize_answer(pred) != gt:
                stats["wrong"] += 1; continue

            # Dedup: same (prompt, completion) pair shouldn't appear twice
            h = hashlib.md5((str(r["prompt_idx"]) + comp).encode()).hexdigest()
            if h in seen_pairs:
                stats["dup_completion"] += 1; continue
            seen_pairs.add(h)

            stats["kept"] += 1
            stats[f"kept_{cat}"] += 1

            keep.append({
                "messages": [
                    {"role": "user",      "content": r["user"]},
                    {"role": "assistant", "content": comp.strip()},
                ],
                "category": cat,
            })

with open(RFT_TRAIN_FILE, "w") as f:
    for r in keep:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Wrote {len(keep)} training examples to {RFT_TRAIN_FILE}\n")
print(f"{'='*60}")
print(f"Stats")
print(f"{'='*60}")
for k in ["total_completions", "no_box", "wrong", "dup_completion", "kept"]:
    if k in stats: print(f"  {k:25s} {stats[k]:>6}")

# Acceptance rate
if stats["total_completions"] > 0:
    rate = 100 * stats["kept"] / stats["total_completions"]
    print(f"\n  Acceptance rate         : {rate:.1f}%  (kept / total)")

print(f"\n{'='*60}")
print(f"Per-category yield")
print(f"{'='*60}")
print(f"  {'category':30s} {'prompts':>8} {'kept':>6} {'avg':>5}")
for cat in sorted(set(k.replace("prompt_","") for k in stats if k.startswith("prompt_"))):
    np = stats[f"prompt_{cat}"]
    nk = stats[f"kept_{cat}"]
    avg = nk / np if np else 0
    print(f"  {cat:30s} {np:>8} {nk:>6} {avg:>5.2f}")


In [ ]:
# ============================================================
# 9. NEXT STEPS — what to do with rft_train.jsonl
# ============================================================
print("""
Next steps (manual, outside this notebook):

1. Download /kaggle/working/rft_train.jsonl

2. Upload as a new Kaggle dataset (name it e.g. 'manish-nemotron-rft-train')

3. In your v76 training notebook:
   - Add the new dataset as input
   - Update CATEGORY_FILES to include 'rft_train.jsonl' for hard categories
     (keep original cipher/unit_conversion/numeral/gravity files for easy
     categories — those CoTs are already clean)
   - Settings:
       LORA_ALPHA   = 64
       LR           = 1e-4    (half of 0.86 run — RFT data is cleaner)
       NUM_EPOCHS   = 1
       WARMUP_STEPS = 50

4. Train one fresh epoch (~6 hrs with fast path)

5. Submit. Expected LB: 0.88-0.90 (vs. 0.86 baseline)

If yield was poor (<30%) on a category, increase K or temperature for
just that category and re-run. The script supports resume.
""")
